In [1]:
import argparse
import sys, os
from pathlib import Path
import requests
import ast
import logging
import openai
import torch
from tqdm import tqdm

from vllm import LLM, SamplingParams
import torch.multiprocessing as mp
from transformers import AutoTokenizer, AutoModel,AutoConfig

mp.set_start_method('spawn', force=True)

logging.basicConfig(level=logging.INFO)
logging.getLogger("vllm").setLevel(logging.WARNING)

sys.path.insert(0, str(Path(".").resolve().parent))
import src.preprocessing.utils.parsingPatients as parsingPatients
import src.preprocessing.utils.chunkPatients as chunkPatients
import src.preprocessing.utils.keywordsPatients as keywordsPatients


/users/jcc2340/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True


# Main

In [2]:
patients_path="../data/coral/"
sentence_tokenizer="en_core_sci_sm"
import spacy
sentence_tokenizer = spacy.load(sentence_tokenizer)
LLM_model="Qwen/Qwen1.5-14B-Chat"



patients_parsed = parsingPatients.parse_raw_clinical_notes(patients_path)
processed_patients, average_chunks_per_patient = chunkPatients.process_patients(patients_parsed,sentence_tokenizer)



/users/jcc2340/.conda/envs/trialmatch/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


## Prompt alternative

If using debug version on **extract_patient_sections_keywords**, initialize server:

```bash
conda activate trialmatch
vllm serve Qwen/Qwen2.5-7B-Instruct --port 8000 --gpu-memory-utilization 0.88
```

In [21]:
keyword_template_path = Path('../src/prompts/extractKeyWords.py')
KEYWORD_PROMPT_TEMPLATE = keywordsPatients.load_KEYWORD_prompt(keyword_template_path)
model=LLM_model

config= {
	'model_name': model,
	'temperature': 0.0,
	'max_tokens': 1024,
    'max_context': 0,
}
def_conf=AutoConfig.from_pretrained(config['model_name'])
config['max_context'] = getattr(def_conf, "max_position_embeddings", None)



In [5]:
# Example usage:
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"  # Ensure 'spawn' method is set for multiprocessing

llm = LLM(
     model=config['model_name'],
                    gpu_memory_utilization=0.88,
                    dtype='bfloat16',
                    max_model_len=13472
)



(EngineCore pid=4034400) INFO 07-07 17:37:48 [core.py:113] Initializing a V1 LLM engine (v0.23.0) with config: model='Qwen/Qwen1.5-14B-Chat', speculative_config=None, tokenizer='Qwen/Qwen1.5-14B-Chat', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=13472, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_en

Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  12% Completed | 1/8 [00:01<00:08,  1.24s/it]
Loading safetensors checkpoint shards:  25% Completed | 2/8 [00:02<00:07,  1.27s/it]
Loading safetensors checkpoint shards:  38% Completed | 3/8 [00:03<00:06,  1.28s/it]
Loading safetensors checkpoint shards:  50% Completed | 4/8 [00:05<00:05,  1.27s/it]
Loading safetensors checkpoint shards:  62% Completed | 5/8 [00:06<00:03,  1.28s/it]
Loading safetensors checkpoint shards:  75% Completed | 6/8 [00:07<00:02,  1.28s/it]
Loading safetensors checkpoint shards:  88% Completed | 7/8 [00:08<00:01,  1.19s/it]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:09<00:00,  1.02it/s]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:09<00:00,  1.15s/it]
(EngineCore pid=4034400) 


(EngineCore pid=4034400) INFO 07-07 17:38:02 [default_loader.py:397] Loading weights took 9.23 seconds
(EngineCore pid=4034400) INFO 07-07 17:38:03 [gpu_model_runner.py:5187] Model loading took 26.43 GiB memory and 11.155243 seconds
(EngineCore pid=4034400) INFO 07-07 17:38:07 [backends.py:1089] Using cache directory: /users/jcc2340/.cache/vllm/torch_compile_cache/58602afa2a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=4034400) INFO 07-07 17:38:07 [backends.py:1148] Dynamo bytecode transform time: 3.87 s
(EngineCore pid=4034400) INFO 07-07 17:38:09 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.409 s
(EngineCore pid=4034400) INFO 07-07 17:38:09 [decorators.py:311] Directly load AOT compilation from path /users/jcc2340/.cache/vllm/torch_compile_cache/torch_aot_compile/235ba7764416c2efa1f16c94d5d5001b38c424a1f970351bd702c803e1a80eea/rank_0_0/model
(EngineCore pid=4034400) INFO 07-07 17:38:09 [monitor.py:53] torch.com

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:06<00:00,  7.46it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:03<00:00, 11.07it/s]


(EngineCore pid=4034400) INFO 07-07 17:38:24 [gpu_model_runner.py:6585] Graph capturing finished in 11 secs, took 0.54 GiB
(EngineCore pid=4034400) INFO 07-07 17:38:24 [gpu_worker.py:639] CUDA graph pool memory: 0.54 GiB (actual), 0.62 GiB (estimated), difference: 0.08 GiB (15.1%).
(EngineCore pid=4034400) INFO 07-07 17:38:24 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=4034400) INFO 07-07 17:38:24 [core.py:306] init engine (profile, create kv cache, warmup model) took 21.63 s (compilation: 5.51 s)
(EngineCore pid=4034400) INFO 07-07 17:38:25 [vllm.py:999] Asynchronous scheduling is enabled.
(EngineCore pid=4034400) INFO 07-07 17:38:25 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


In [23]:
import subprocess, sys
import io
from contextlib import redirect_stderr
import importlib 
importlib.reload(keywordsPatients)
f = io.StringIO()
try:
    keywordsPatients.extract_patient_sections_keywords(
        processed_patients[32], n_keywords=5, template=KEYWORD_PROMPT_TEMPLATE,
        config=config, type_run='hpc', llm=llm
    )
except Exception as e:
    print("CAUGHT:", e)

CUDA available: True


Processed prompts: 100%|██████████| 12/12 [00:08<00:00,  1.36it/s, est. speed input: 713.28 toks/s, output: 141.00 toks/s]

['```json', '{', '  "KEYWORDS": ["breast cancer", "left breast", "Stage IIb", "HER-2-neu neg", "BRCA test negative"],', '  "KEYWORD_EXTRACTION_REASONING": "The keywords were chosen because they directly relate to the patient\'s diagnosis (breast cancer, left breast, Stage IIb), her specific cancer subtype (HER-2-neu neg), the genetic testing result (BRCA test negative), and the treatment context (Stage IIb stage)."', '}', '```']
Extracted keywords for section _preamble: ['breast cancer', 'left breast', 'Stage IIb', 'HER-2-neu neg', 'BRCA test negative']
['```json', '{', '  "KEYWORDS": ["breast cancer", "left breast", "March 2017"],', '  "KEYWORD_EXTRACTION_REASONING": "The keywords are explicitly stated and directly related to the patient\'s diagnosis. \'Breast cancer\' is the primary condition, \'left breast\' specifies the location, and \'March 2017\' indicates the diagnosis date, which is relevant for tracking the disease progression."', '}', '```']
Extracted keywords for section pa

In [66]:
test_set=processed_patients[32:40]

for p in tqdm(test_set, desc="Generating keywords", unit="patient", dynamic_ncols=True):
	# For each patient send all sections to the LLM for keyword extraction. Sections are stored in a list section_jobs
	keywordsPatients.extract_patient_sections_keywords(
					p, n_keywords=5, template=KEYWORD_PROMPT_TEMPLATE, 
					config=config, type_run='hpc', llm=llm
					)

-----
<|im_start|>system
You are an expert in clinical trial eligibility criteria. Output only valid JSON, no extra text or explanations.<|im_end|>
<|im_start|>user
You are a highly experienced clinical assistant specializing in medical terminology extraction from clinical notes.

## Task
Extract the 5 most clinically relevant keywords from the provided patient note section, ranked by importance (most important first).
If a genetic biomarker is mentioned, you must include it as a keyword, always prioritizing it over other keywords.

INPUT: HPI: ***** ***** ***** is a 58 y. o. female here in consultation to discuss treatment options for her newly diagnosed breast cancer whose history is as follows: Patient Active Problem List Diagnosis Date Noted .  Breast cancer, left breast 06/24/2017 Stage IIb (T2N1M0) Gr 2 IDC (ER pos/PR pos/Her-2-neu neg) March 2017 patient found a left breast mass measuring 2cm after she felt a pinch in her left breast Saw primary care doctor 04/10/17 BL Dx mammog


Rendering prompts:   0%|          | 0/12 [00:00<?, ?it/s]


Error during keyword extraction: EngineCore encountered an issue. See stack trace (above) for the root cause.
-----
<|im_start|>system
You are an expert in clinical trial eligibility criteria. Output only valid JSON, no extra text or explanations.<|im_end|>
<|im_start|>user
You are a highly experienced clinical assistant specializing in medical terminology extraction from clinical notes.

## Task
Extract the 5 most clinically relevant keywords from the provided patient note section, ranked by importance (most important first).
If a genetic biomarker is mentioned, you must include it as a keyword, always prioritizing it over other keywords.

INPUT: ID: ***** ***** is a 33 y. o. premenopausal patient with a recent diagnosis of a HR+ metastatic breast cancer involving lymph nodes, who presents in consultation to discuss treatment options and to establish care. HPI: The patient was evaluated by her PCP on 10/30/16 at which time she reported a new right lateral neck lump. Her exam was notab


Rendering prompts:   0%|          | 0/17 [00:00<?, ?it/s]


Error during keyword extraction: EngineCore encountered an issue. See stack trace (above) for the root cause.
-----
<|im_start|>system
You are an expert in clinical trial eligibility criteria. Output only valid JSON, no extra text or explanations.<|im_end|>
<|im_start|>user
You are a highly experienced clinical assistant specializing in medical terminology extraction from clinical notes.

## Task
Extract the 5 most clinically relevant keywords from the provided patient note section, ranked by importance (most important first).
If a genetic biomarker is mentioned, you must include it as a keyword, always prioritizing it over other keywords.

INPUT: SUBJECTIVE: This is a very pleasant 47-year-old African-American gentleman with a diagnosis of locally advanced adenocarcinoma of the pancreas. He presented with several months of vague abdominal discomfort and had some gross hematuria in September. He went to ***** Hospital and a CT scan was done which revealed a pancreatic mass. His hematur

Generating keywords:  25%|██▌       | 2/8 [00:17<00:53,  8.85s/patient]


KeyboardInterrupt: 

## Embeddings and vector stores

In [10]:
import numpy as np
import faiss
from transformers import AutoTokenizer, AutoModel

# Information retrieval Contrastively Pretrained Transformer model for zero-shot semantic IR in biomedicine
# Bi encoder architecture, first retriever (query encoder and document encoder), second re-ranker.
# It was trained on 255M query-article pairs from Pubmed search logs. Real query + the article the user clicked on.
	#the two encoders are trained togheter so their outputs land in a shared embeding space (the same query is also fed with do´cuments that the user did not click on, so the model learns to distinguish between relevant and non-relevant documents)
     #Because this happens at a massive scale the model learns to scale achross real miomedical behavior, the resulting embedings tend to organize around genuine biomedical relationshipds
    # The text is first tokenized and then passed to the model 
# The second component or reranker takes the top candidates retrieved by the encoder (embedding similaruty) and re-scores them more precisely using token level interactions.  It is trained with
# the negative distrivution  sampled from the pretrained MedCPT retriever
# The retriever takes out all "Keyword-match article" because is relatively easy to match them, it needs to understand the semantic understanding of the query and the document to be able to match them. The reranker is trained with a more difficult negative distribution, which is sampled from the pretrained MedCPT retriever. 



#the weights of those transformer layers were shaped by the contrastive training to do something specific: encode the input such that semantically/clinically related texts land near each other and unrelated ones land far apart. The "knowledge" isn't stored as a lookup of phrases to locations — it's stored in the weights of the network, which then compute an appropriate vector for any new text you give it, including text it never saw during training.
# That's the part that makes it generalize: if you feed it a clinical phrase it never saw verbatim during training, it still produces a sensible vector, because the network learned general patterns of biomedical language

#could include bert or medcpt but bert shows worse performance based on MedCPT paper 
# MedCPT: https://huggingface.co/ncbi/MedCPT-Query-Encoder
# Check alternatives on references https://academic.oup.com/bioinformatics/article/39/11/btad651/7335842

# This is the tokek=nizer and model for MedCPT 
tokenizer_med=AutoTokenizer.from_pretrained("ncbi/MedCPT-Query-Encoder")
model_med=AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder")

INFO:faiss.loader:Loading faiss with AVX512 support.
INFO:faiss.loader:Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1849.64it/s]


In [11]:
# FAISS version # may send to graveyard
def build_vector_stores_for_patient(patient:Patient, tokenizer: AutoTokenizer,model: AutoModel,option: str) -> dict:
	'''Builds a vector store for a patient by embedding their clinical note sections and storing the embeddings in a FAISS index.
	:param patient: Patient object with extracted sections and chunks
	:param tokenizer: the tokenizer to use for embedding
	:param model: the model to use for embedding
	:param option: String indicating the option for vector store creation, it can be chunks or keywords
	:return: index and metadata for the vector store based on the specified option'''
	''' One index per patient, which includes multiple embeddings (one per chunk or keyword)
	If a patient has 3 sections with 2 chunks each we should have 6 embeddings in the index, and the metadata should reflect which section and chunk each embedding corresponds to, for example:
	index vector 0  →  chunk_metadata[0] = {patient_id, section: "past medical history", chunk: "..."}
	index vector 1  →  chunk_metadata[1] = {patient_id, section: "past medical history", chunk: "..."}
	index vector 2  →  chunk_metadata[2] = {patient_id, section: "family history", chunk: "..."}
	...
	'''
	
	threshold,dimension = 0.96,768
	patient_embeddings = []
	if option == 'chunks':
		chunk_metadata = [] # we can keep track of the metadata for each chunk, such as the section it came from and the original text, to help with debugging and analysis later on
		try:
			index = faiss.IndexFlatIP(dimension)
			for section in patient.PatientSections:
				for chunk in section.chunks:
					embedding = embed_text(chunk, tokenizer, model) #chunks are completely filtered before this step
					if len(patient_embeddings) > 0:
						existing_embeddings = np.array(patient_embeddings)
						norm_embedding = embedding / np.linalg.norm(embedding)
						norm_existing_embeddings = existing_embeddings / np.linalg.norm(existing_embeddings, axis=1, keepdims=True)
						cosine_similarities = norm_existing_embeddings @ norm_embedding
						if np.any(cosine_similarities > threshold):
							print("Skipping chunk due to similarity")
							continue
					patient_embeddings.append(embedding)
					#What information are we keeping track of?
					chunk_metadata.append({
						"patient_id": patient.patient_id,
						"section": section.section_name,
						"chunk": chunk,
					})
			if patient_embeddings:
				patient_embeddings = np.array(patient_embeddings)
				index.add(patient_embeddings)
				
			return index, chunk_metadata
		except Exception as e:
			print(f"Error building vector store for patient {patient.patient_id}: {e}")

	elif option == 'keywords':
		keyword_metadata = []
		try:
			index = faiss.IndexFlatIP(dimension)
			for s in patient.PatientSections:
				if s.keywords is not None:
					for k in s.keywords:
						embedding = embed_text(k, tokenizer, model)
						if len(patient_embeddings) > 0:
							existing_embeddings = np.array(patient_embeddings)
							norm_embedding = embedding / np.linalg.norm(embedding)
							norm_existing_embeddings = existing_embeddings / np.linalg.norm(existing_embeddings, axis=1, keepdims=True)
							cosine_similarities = norm_existing_embeddings @ norm_embedding
							if np.any(cosine_similarities > threshold):
								print("Skipping keyword due to similarity")
								continue
						patient_embeddings.append(embedding)
						keyword_metadata.append({
							"patient_id": patient.patient_id,
							"section": s.section_name,
							"keyword": k,
						})
			if patient_embeddings:
				patient_embeddings = np.array(patient_embeddings)
				index.add(patient_embeddings)
			return index, keyword_metadata
		except Exception as e:
			print(f"Error building vector store for patient {patient.patient_id}: {e}")

In [12]:
@dataclass
class PatientVectorStore:
	"""Class for storing patient embeddings and metadata in a FAISS index."""
	index: faiss.IndexFlatIP
	metadata: List[dict]
	n_chunks: int

In [13]:
import chromadb
from chromadb import Collection
import pickle

In [ ]:

def embed_text(text: str,tokenizer=tokenizer_med, model=model_med) -> np.ndarray:
    '''Embed a string of text [chunk]
    :param text: str - the text to be embedded
	:param tokenizer: the tokenizer to use for embedding, defaults to the MedCPT tokenizer
	:param model: the model to use for embedding, defaults to the MedCPT model
    :return: numpy array representing the embedding of the input text float32
    '''
    with torch.no_grad():
        inputs = tokenizer(text, return_tensors="pt", 
                           truncation=True, padding=True,
                           max_length=512)
        embeddings = model(**inputs).last_hidden_state[:,0,:].squeeze().numpy() # we take the embedding of the [CLS] token, which is the first token in the input sequence
    return embeddings.astype('float32')

def _add_keywords(patient:Patient, tokenizer: AutoTokenizer, model: AutoModel,collection: Collection) -> None:
	'''embedding keywords and adding them to the ChromaDB collection for a patient
	:param patient: Patient object with extracted sections and keywords
	:param tokenizer: the tokenizer to use for embedding
	:param model: the model to use for embedding
	:param collection: ChromaDB collection to add the embeddings to'''

	threshold = 0.96
	#embedding tracks embeddings, documents tracks the text of the keyword, metadata tracks the patient id and section name, ids tracks the unique id for each keyword
	embeddings, documents,metadata,ids_list = [], [], [], []
	try:
		for s in patient.PatientSections:
			if not s.keywords:
				continue
			for i, kw in tqdm(enumerate(s.keywords), desc=f"Processing keywords for patient {patient.patient_id}", unit="keyword"):
				e = embed_text(kw, tokenizer, model)
				if len(embeddings) > 0:
					existin_emb = np.array(embeddings)
					norm_embedding = e / np.linalg.norm(e)
					norm_existing_embeddings = existin_emb / np.linalg.norm(existin_emb, axis=1, keepdims=True)
					cosine_similarities = norm_existing_embeddings @ norm_embedding
					if np.any(cosine_similarities > threshold):
						print(f"Skipping keyword due to similarity: {kw}")
						continue
				embeddings.append(e.tolist())
				documents.append(kw)
				metadata.append({
					"patient_id": patient.patient_id,
					"section": s.section_name,})
				ids_list.append(f"{patient.patient_id}_{s.section_name}_kw_{i}")
		if embeddings:
			collection.add(ids=ids_list, embeddings=embeddings, documents=documents, metadatas=metadata)

	except Exception as e:
		print(f"Error adding keywords for patient {patient.patient_id}: {e}")

				
			
def _add_chunks(patient:Patient, tokenizer: AutoTokenizer, model: AutoModel,collection: Collection) -> None:
	'''embedding chunks and adding them to the ChromaDB collection for a patient
	:param patient: Patient object with extracted sections and chunks
	:param tokenizer: the tokenizer to use for embedding
	:param model: the model to use for embedding
	:param collection: ChromaDB collection to add the embeddings to'''
	threshold = 0.96
	#embedding tracks embeddings, documents tracks the text of the chunk, metadata tracks the patient id and section name, ids tracks the unique id for each chunk
	embeddings, documents,metadata,ids_list = [], [], [], []
	try:
		for s in patient.PatientSections:
			if not s.chunks:
				continue
			for i, chunk in tqdm(enumerate(s.chunks), desc=f"Processing chunks for patient {patient.patient_id}", unit="chunk"):
				e = embed_text(chunk, tokenizer, model)
				if len(embeddings) > 0:
					existin_emb = np.array(embeddings)
					norm_embedding = e / np.linalg.norm(e)
					norm_existing_embeddings = existin_emb / np.linalg.norm(existin_emb, axis=1, keepdims=True)
					cosine_similarities = norm_existing_embeddings @ norm_embedding
					if np.any(cosine_similarities > threshold):
						print(f"Skipping chunk due to similarity: {chunk}")
						continue
				embeddings.append(e.tolist())
				documents.append(chunk)
				metadata.append({
					"patient_id": patient.patient_id,
					"section": s.section_name,})
				ids_list.append(f"{patient.patient_id}_{s.section_name}_chunk_{i}")
		if embeddings:
			collection.add(ids=ids_list, embeddings=embeddings, documents=documents, metadatas=metadata)

	except Exception as e:
		print(f"Error adding chunks for patient {patient.patient_id}: {e}")

def generate_chromaDB(processed_patients: List[Patient], tokenizer: AutoTokenizer, model: AutoModel, save_dir: Path, option: str) -> None:
	"""Generates a ChromaDB database for the processed patients, storing both chunks and keywords.
	:param processed_patients: List of Patient objects with extracted sections and chunks
	:param tokenizer: the tokenizer to use for embedding
	:param model: the model to use for embedding
	:param save_dir: Path to the directory where the ChromaDB database will be saved, do not include the database name, just the directory
	:param option: String indicating the option for vector store creation, it can be 'chunks'. 'keywords' or 'both'

	The chromaDB database will contain on save_dir/chromaDB_patients: 
	- pkl file with the processed patients
	- chunks collection with embeddings and metadata for each chunk (chunk text + embedding + patient id + section name)
	- keywords collection with embeddings and metadata for each keyword (keyword text + embedding + patient id + section name)
	"""
	from datetime import datetime
	timestamp = datetime.now().strftime("%Y%m%d")

	save_path = Path(save_dir)
	if not save_path.exists():
		save_path.mkdir(parents=True, exist_ok=True)
	with open(save_path / f"processed_patients_{timestamp}.pkl", "wb") as f:
		pickle.dump(processed_patients, f)

	

	client = chromadb.PersistentClient(path=str(save_dir / "chromaDB_patients"))
	if option not in ["chunks", "keywords", "both"]:
		raise ValueError("Invalid option. Must be 'chunks', 'keywords', or 'both'.")
	if option in ["chunks", "both"]:
		chunks_collection = client.get_or_create_collection("chunks", metadata={"hnsw:space": "ip"})
		print(f"Created ChromaDB collection for chunks at {save_dir  / 'chunks'}")
		existing_patients_with_chunks = set(chunks_collection.get()['ids'])#collections of strings with chunkid_section_patientid
	if option in ["keywords", "both"]:
		keywords_collection = client.get_or_create_collection("keywords", metadata={"hnsw:space": "ip"})
		print(f"Created ChromaDB collection for keywords at {save_dir / 'keywords'}")
		existing_patients_with_keywords = set(keywords_collection.get()['ids'])




	patients_to_add = []

	for patient in tqdm(processed_patients, desc="Building ChromaDB", unit="patient"):
		# This patient chunks or keywords already exist in the ChromaDB collection, so we skip it to avoid duplicates

		#It is skipping chunks if the keyword already exists, and vice versa. This is because we are using the patient_id as the unique identifier for both chunks and keywords. If a patient has both chunks and keywords, we want to skip adding them if either already exists in the collection. This is to avoid duplicates and ensure that we only have one entry per patient in the collection.
		if any (idp.startswith(str(patient.patient_id)) for idp in existing_patients_with_chunks ) and option in ["chunks", "both"]:
			print(f"Chunk {patient.patient_id} already exists in chunks collection. Skipping.")
			continue
		if any (idp.startswith(str(patient.patient_id)) for idp in existing_patients_with_keywords) and option in ["keywords", "both"]:
			print(f"Keyword {patient.patient_id} already exists in keywords collection. Skipping.")
			continue

		if option in ["keywords", "both"]:
			print(f"Adding keywords for patient {patient.patient_id} to ChromaDB collection.")
			_add_keywords(patient, tokenizer, model, keywords_collection)
		if option in ["chunks", "both"]:
			print(f"Adding chunks for patient {patient.patient_id} to ChromaDB collection.")
			_add_chunks(patient, tokenizer, model, chunks_collection)
	return client

In [ ]:
#Load and query

def _load_patientDB(save_dir: Path):
	"""Loads the ChromaDB database for the processed patients.
	:param save_dir: Path to the directory where the ChromaDB database is saved
	:return: ChromaDB client object
	"""
	client = chromadb.PersistentClient(path=str(save_dir / "chromaDB_patients"))
	with open(save_dir / "processed_patients.pkl", "rb") as f:
		processed_patients = pickle.load(f)
	return client, processed_patients


```markdown
P_cind + Chunk Metadata = One patient
P_kind + key metadata = One patient
```

In [31]:
generate_patientDB = False  # Set to True to generate the ChromaDB database, False to load an existing one
if generate_patientDB:
	client = generate_chromaDB(test_set, tokenizer_med, model_med, save_dir=Path("../database/chromadb"), option="both")

if Path("../database/chromadb").exists() and not generate_patientDB:
	client, processed_patients = _load_patientDB(save_dir=Path("../database/chromadb"))
	print(f"Loaded ChromaDB database from {Path('../database/chromadb')}")

Loaded ChromaDB database from ../database/chromadb


## Save patients and embedings in a database

# Synthetic

In [33]:
example="/users/jcc2340/g2lab/projects/precision_oncology/data/synthetic/patients/TREC_2022/topics2022.xml"
sigir="/users/jcc2340/g2lab/projects/precision_oncology/data/synthetic/patients/SIGIR/topics-2014_2015-description.topics"

patients=parse_trec_topics(example) # dictionary of dictionaries, where the key is the patient id and the value is a dictionary with keys "id", "query", and "source"
sigir_patients=parse_sigir_topics(sigir)
for i in range(0, len(patients)):
	print(f"Gender: {patients[i].gender}")

Gender: Gender.male
Gender: Gender.female
Gender: Gender.male
Gender: Gender.female
Gender: Gender.male
Gender: Gender.male
Gender: Gender.female
Gender: Gender.male
Gender: Gender.female
Gender: Gender.female
Gender: Gender.male
Gender: Gender.male
Gender: Gender.male
Gender: Gender.male
Gender: Gender.male
Gender: Gender.female
Gender: Gender.male
Gender: Gender.male
Gender: Gender.female
Gender: Gender.male
Gender: Gender.male
Gender: Gender.male
Gender: Gender.female
Gender: Gender.male
Gender: Gender.female
Gender: Gender.female
Gender: Gender.female
Gender: Gender.female
Gender: Gender.male
Gender: Gender.female
Gender: Gender.female
Gender: Gender.male
Gender: Gender.male
Gender: Gender.male
Gender: Gender.female
Gender: Gender.female
Gender: Gender.male
Gender: Gender.male
Gender: Gender.male
Gender: Gender.female
Gender: Gender.male
Gender: Gender.female
Gender: Gender.female
Gender: Gender.male
Gender: Gender.female
Gender: Gender.male
Gender: Gender.female
Gender: Gender.mal

In [23]:
for i in range(0, len(patients)):
    print(patients[i].unique_id)
    print(patients[i].age)
    print(patients[i].gender)
    print(patients[i].current_medical_history)
    print(patients[i].past_medical_history)
    print(patients[i].family_medical_history)

topics2022_1
19
Gender.unknown
None
None
None
topics2022_2
32
Gender.unknown
A 32-year-old woman comes to the hospital with vaginal spotting.  Her last menstrual period was 10 weeks ago. She has regular menses lasting for 6 days and repeating every 29 days She has multiple male partners, and she is inconsistent with using barrier contraceptives. Vital signs are normal.  Serum β-hCG level is 1800 mIU/mL, and a repeat level after 2 days shows an abnormal rise to 2100 mIU/mL.  Pelvic ultrasound reveals a thin endometrium with no gestational sac in the uterus.
. Medical history is significant for appendectomy and several complicated UTIs.
None
topics2022_3
51
Gender.unknown
None
None
None
topics2022_4
66
Gender.unknown
A 66-year-old woman comes to the office due to joint pain in the hands and periodic morning stiffness that lasts less than 15 minutes.  The pain is moderately severe and worsens with daily activity. The patient used Tylenol with minimal relief  Physical examination shows fir

In [13]:
print(patients[0].)

None
